# HOMEWORK: Agentic RAG

## PREPARATION

In [1]:
%load_ext autoreload

%autoreload 2

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

### How many lesson pages

In [4]:
len(documents)

72

Answer Q1: 72

In [5]:
documents[2]

{'content': '# RAG\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=JktYwBIDErk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nWe run free Zoomcamp courses at DataTalks.Club on data engineering,\nmachine learning, MLOps, and other topics. Each course has its own\nFAQ document with common questions and answers.\n\nSome of these documents have over 300 questions. Students ask us\nthings in Slack like "Can I still join after the course started?" or\n"How do I get a certificate?" Finding those answers in the FAQ is\ntedious.\n\nWe want a bot that takes all this knowledge and answers student\nquestions in natural language.\n\nIn this module, we\'ll build that system. But first, let\'s see why we\ncan\'t send the question straight to an LLM and call it a day.\n\n## Plain LLMs lack our data\n\nFirst, let\'s define a function to talk to the LLM:\n\n```python\ndef llm(prompt):\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=prompt\n    )\n

### SEARCH

Se implemento ingest.py para la generacion de la data y la data indexada

In [6]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [7]:
question = 'How does the agentic loop keep calling the model until it stops?'

search_results = index.search(
    question,
    num_results=5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

Answer Q2: 01-agentic-rag/lessons/14-agentic-loop.md

### RAG

Se actualizo rag_helper.py en los metodos search y build_context para la base de conocimientos manejada.

In [8]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [9]:
from rag_helper import RAGBase
assistant = RAGBase(llm_client=openai_client, index=index)

In [10]:
question = 'How does the agentic loop keep calling the model until it stops?'
answer, usage = assistant.rag(question)

In [11]:
answer

'It keeps a `while True` loop around the model call.\n\nAfter each response, the code checks whether the model returned any `function_call` items:\n\n- if yes, it runs those tools, appends the tool outputs to `messages`, and loops again\n- if no, it `break`s\n\nSo the stop condition is simply: **no function calls in the latest response**.'

In [12]:
usage

ResponseUsage(input_tokens=7121, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=83, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7204)

Answer Q3: 7121 Imput (prompt) tokens

### CHUNKING

In [13]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [14]:
len(chunks)

295

Answer Q4: 295

### RAG with chunking

In [15]:
chunks[3]

{'start': 0,
 'content': '# Environment\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nFor this module, all you need is Python with Jupyter.\n\n## Prerequisites\n\nYou need the following:\n\n- Python (3.14 or later)\n- An [OpenAI account](https://openai.com/) (or an OpenAI-compatible\n  provider like Groq, Gemini, or Ollama)\n- Basic familiarity with Python and the command line\n\n## Creating the project\n\nWe\'ll start from scratch - no cloning needed. You\'ll create the\nproject yourself, step by step.\n\nFirst, install uv. It\'s a Python package manager, and I switched all my\nprojects to it because it\'s fast and convenient. Once I started using\nit, I never wanted to go back.\n\nOn Mac or Linux:\n\n```bash\ncurl -LsSf https://astral.sh/uv/install.sh | sh\n```\n\nOn Windows:\n\n```powershell\npowershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"\n```\n\n(You can also use `pip install

In [16]:
index_chunks = build_index(chunks)

assistant_chunks = RAGBase(llm_client=openai_client, index=index_chunks)

In [17]:
question = 'How does the agentic loop keep calling the model until it stops?'
answer_chunks, usage_chunks = assistant_chunks.rag(question)

In [18]:
answer_chunks

'It keeps calling the model inside a `while True` loop.\n\nEach round:\n- the model is called with the current message history,\n- any tool/function calls are executed,\n- their outputs are added back into `messages`,\n- and a flag `has_function_calls` tracks whether the model asked for tools.\n\nThe loop stops when the model returns a response with **no function calls**:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nSo the agent keeps looping until the model gives a final answer without requesting any more tools.'

In [19]:
usage_chunks

ResponseUsage(input_tokens=2304, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=115, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2419)

In [20]:
ratio_fewer = usage.input_tokens / usage_chunks.input_tokens

print("How many fewer input tokens does the chunked version send?, It was almost ", ratio_fewer)

How many fewer input tokens does the chunked version send?, It was almost  3.0907118055555554


Answer Q5: Imput tokens = 2304, almost 3x fewer

### TURNING INTO AN AGENT

In [22]:
# Define search tools
# Reference: Example FAQ Search Agent https://github.com/alexeygrigorev/toyaikit

from typing import List, Dict, Any

class SearchTools:
    def __init__(self, index):
        self.index = index

    def search(self, query: str) -> List[Dict[str, Any]]:
        results = self.index.search(
            query=query,
            num_results=5,
        )
        return results



In [25]:
search_tools = SearchTools(index_chunks)

In [28]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat import IPythonChatInterface
from toyaikit.llm import OpenAIClient

In [30]:
tools = Tools()
tools.add_tools(search_tools)

developer_prompt = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""

openai_client = OpenAIClient(
    model="gpt-5.4-mini",
    client=OpenAI()
)

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=IPythonChatInterface(),
    llm_client=OpenAIClient()
)

runner.run()

-> Response received


-> Response received


KeyboardInterrupt: 